# Анализ документов _DOCUMENT138 и _DOCUMENT137

Используем ПРАВИЛЬНУЮ архитектуру из @1c.refactoring.md:
- DocumentExtractor для документов
- НЕ FlatTableExtractor (это legacy подход)


In [15]:
import sys
import os
import pandas as pd
sys.path.append('..')

# ИСПОЛЬЗУЕМ РЕАЛЬНО СУЩЕСТВУЮЩИЕ КОМПОНЕНТЫ ХОРОШЕГО КАЧЕСТВА
from src.extractors.simple_document_extractor import SimpleDocumentExtractor
from src.processors.database_connector import DatabaseConnector
from src.processors.blob_processor import BlobProcessor
from onec_dtools.database_reader import DatabaseReader


from src.processors.blob_processor import BlobProcessor
# Создаем экстрактор и blob процессор


# Создаем РЕАЛЬНО СУЩЕСТВУЮЩИЕ компоненты хорошего качества
db_connector = DatabaseConnector('../data/raw/1Cv8.1CD')
db_connector.connect()

print('✅ Подключение к базе данных установлено')
print(f'📊 База данных: {db_connector.file_path}')


✅ Простой патч успешно применен к библиотеке onec_dtools
📋 Добавлена поддержка всех типов полей включая VB
✅ Подключение к базе данных установлено
📊 База данных: ../data/raw/1Cv8.1CD


In [ ]:
# Используем РЕАЛЬНО СУЩЕСТВУЮЩИЕ компоненты хорошего качества
print('🔍 АНАЛИЗ ДОКУМЕНТОВ ЧЕРЕЗ SimpleDocumentExtractor')
print('=' * 70)

try:
    
    # Создаем SimpleDocumentExtractor (РЕАЛЬНО СУЩЕСТВУЕТ!)
    document_extractor = SimpleDocumentExtractor(db_connector)
    
    print(f'✅ База данных подключена!')
    print(f'📊 Всего таблиц в базе: {len(db_connector.db_reader.tables):,}')
    
    # Анализируем _DOCUMENT138 через SimpleDocumentExtractor
    print(f'\n📋 АНАЛИЗ _DOCUMENT138 (Поступление товаров):')
    documents_138 = document_extractor.extract_documents('_DOCUMENT138', limit=3)
    print(f'  📊 Извлечено документов: {len(documents_138)}')
    
    for i, doc in enumerate(documents_138):
        print(f'    Документ {i+1}:')
        for key, value in list(doc.items())[:5]:  # Первые 5 полей
            if isinstance(value, bytes):
                print(f'      {key}: [BLOB {len(value)} байт]')
            else:
                print(f'      {key}: {value}')
    
    # Анализируем _DOCUMENT137 через SimpleDocumentExtractor
    print(f'\n📋 АНАЛИЗ _DOCUMENT137 (Розничные продажи):')
    documents_137 = document_extractor.extract_documents('_DOCUMENT137', limit=3)
    print(f'  📊 Извлечено документов: {len(documents_137)}')
    
    for i, doc in enumerate(documents_137):
        print(f'    Документ {i+1}:')
        for key, value in list(doc.items())[:5]:  # Первые 5 полей
            if isinstance(value, bytes):
                print(f'      {key}: [BLOB {len(value)} байт]')
            else:
                print(f'      {key}: {value}')
        
except Exception as e:
    print(f'❌ Ошибка анализа: {e}')
    import traceback
    traceback.print_exc()


🔍 АНАЛИЗ ДОКУМЕНТОВ ЧЕРЕЗ DocumentExtractor
✅ Простой патч успешно применен к библиотеке onec_dtools
📋 Добавлена поддержка всех типов полей включая VB
❌ Ошибка анализа: name 'DocumentExtractor' is not defined


Traceback (most recent call last):
  File "/var/folders/wm/r_fhn76x4hx7xgc2jkzpn1d40000gn/T/ipykernel_27640/4157049751.py", line 11, in <module>
    document_extractor = DocumentExtractor(db_connector)
                         ^^^^^^^^^^^^^^^^^
NameError: name 'DocumentExtractor' is not defined


In [ ]:
# Создаем плоскую таблицу и сохраняем в Parquet
print('💾 Создание плоской таблицы и сохранение в Parquet...')

try:
    # Создаем плоскую таблицу через новый метод
    flat_table = extractor.extract_flat_table()
    print(f'✅ Создана плоская таблица: {len(flat_table)} записей')
    
    # Сохраняем в Parquet
    if flat_table:
        df = pd.DataFrame(flat_table)
        parquet_file = 'data/results/flat_table_new_extractors.parquet'
        df.to_parquet(parquet_file, index=False)
        print(f'✅ Сохранено в Parquet: {parquet_file}')
        print(f'📊 Размер файла: {os.path.getsize(parquet_file) / 1024 / 1024:.2f} MB')
    else:
        print('❌ Нет данных для сохранения')
        
except Exception as e:
    print(f'❌ Ошибка сохранения: {e}')


In [ ]:
# Показываем результаты как DataFrame
print('📊 СОЗДАНИЕ ПЛОСКОЙ ТАБЛИЦЫ С МАППИНГОМ ПОЛЕЙ')
print('=' * 70)

# Создаем общий DataFrame из всех результатов
all_data = []
for table_name, records in results.items():
    for record in records:
        # Создаем плоскую запись
        flat_record = {
            'table_name': table_name,
            'row_index': record.get('row_index', 0),
            'id': record.get('id', '')
        }
        
        # Добавляем поля
        for key, value in record.get('fields', {}).items():
            flat_record[f'field_{key}'] = value
        
        # Добавляем BLOB поля
        for key, blob_data in record.get('blobs', {}).items():
            if 'value' in blob_data and 'content' in blob_data['value']:
                content = blob_data['value']['content']
                if content and len(content.strip()) > 0:
                    flat_record[f'blob_{key}_content'] = content[:100]  # Первые 100 символов
                else:
                    flat_record[f'blob_{key}_content'] = f'[BLOB {blob_data.get("size", 0)} байт]'
        
        all_data.append(flat_record)

# Создаем DataFrame
if all_data:
    df = pd.DataFrame(all_data)
    print(f'✅ Создана плоская таблица: {df.shape[0]} строк, {df.shape[1]} колонок')
    
    # Показываем первые 5 строк
    print('\n📋 ПЕРВЫЕ 5 СТРОК ПЛОСКОЙ ТАБЛИЦЫ:')
    display_df = df.head(5).iloc[:, :8]  # Первые 8 колонок
    print(display_df.to_string(index=True, max_cols=8, max_colwidth=20))
    
    # Статистика
    print(f'\n📈 СТАТИСТИКА:')
    print(f'  📊 Всего записей: {len(df):,}')
    print(f'  📋 Колонок: {len(df.columns)}')
    print(f'  📄 Таблиц: {df["table_name"].nunique()}')
    
    # Показываем размеры файлов
    parquet_files = [f for f in os.listdir('.') if f.endswith('.parquet')]
    if parquet_files:
        print(f'\n💾 СОЗДАННЫЕ PARQUET ФАЙЛЫ:')
        for file in parquet_files:
            size_mb = os.path.getsize(file) / 1024 / 1024
            print(f'  📄 {file}: {size_mb:.2f} MB')
else:
    print('❌ Нет данных для отображения')
